# CAPM LP1 Analysis - Clean Notebook
## Capital Asset Pricing Model: Microsoft, GE, Ford (1993-2003)

This notebook runs the complete CAPM analysis pipeline using modular Python scripts from the `src/` directory.

## Setup and Configuration

In [ ]:
# Import standard libraries
import sys
from pathlib import Path

# Add src to path for imports
project_root = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
sys.path.insert(0, str(project_root / 'src'))

print(f"Project root: {project_root}")
print(f"Python path includes: {project_root / 'src'}")

In [ ]:
# Import configuration
from config import (
    ROOT_DIR, DATA_DIR, REPORT_DIR, FIG_DIR, TABLE_DIR,
    TRADING_DAYS, ASSETS, MARKET_TICKER, ensure_dirs
)

# Ensure output directories exist
ensure_dirs()

print("\n=== Configuration ===")
print(f"Root directory: {ROOT_DIR}")
print(f"Data directory: {DATA_DIR}")
print(f"Report directory: {REPORT_DIR}")
print(f"Assets: {ASSETS}")
print(f"Market ticker: {MARKET_TICKER}")
print(f"Trading days per year: {TRADING_DAYS}")

## Step 1: Load and Clean Data

Load daily price data from CSV, parse dates, remove missing values, and prepare for analysis.

In [ ]:
from load_and_clean import load_and_clean

# Load data from CSV
df = load_and_clean(source='csv')

print(f"\nData shape: {df.shape}")
print(f"Date range: {df['Date'].min()} to {df['Date'].max()}")
print(f"\nFirst few rows:")
print(df.head())

In [ ]:
# Display data info
print("\nData Info:")
print(df.info())
print(f"\nColumn names: {list(df.columns)}")

## Step 2: Compute Returns and Excess Returns

Calculate daily log returns and excess returns using:
- Daily log returns: $R_t = \ln(P_t / P_{t-1})$
- Daily risk-free rate: $R_f^{daily} = (Tbill/100) / 252$
- Excess returns: $R^{excess} = R_t - R_f^{daily}$

In [ ]:
from returns_and_excess import compute_returns_and_excess

# Compute all returns and excess returns
df = compute_returns_and_excess(df)

print(f"\nData shape after returns computation: {df.shape}")
print(f"\nColumns with returns and excess returns:")
return_cols = [col for col in df.columns if 'Return' in col or 'Excess' in col]
print(return_cols)

In [ ]:
# Display sample of computed returns
print("\nSample of returns (first 5 rows):")
cols_to_show = ['Date', 'Msft_Return', 'SP500_Return', 'Rf_daily', 'Msft_Excess', 'SP500_Excess']
print(df[cols_to_show].head(10))

print("\nDescriptive statistics of excess returns:")
excess_cols = ['Msft_Excess', 'GE_Excess', 'Ford_Excess', 'SP500_Excess']
print(df[excess_cols].describe())

## Step 3: CAPM Regression Analysis

Run OLS regressions for each asset:
$$R_i^{excess} = \alpha + \beta \cdot R_m^{excess} + \epsilon$$

**Hypothesis Test:**
- H₀: α = 0 (Security is correctly priced)
- H₁: α ≠ 0 (Security is mispriced)
- Significance level: 5%

In [ ]:
from regressions import capm_regression_all_assets, save_regression_results

# Run CAPM regressions for all assets
results_df, models = capm_regression_all_assets(df)

# Save results to CSV
save_regression_results(results_df)

In [ ]:
# Display results summary
print("\n=== CAPM REGRESSION SUMMARY ===")
summary_cols = ['Asset', 'Alpha', 'Alpha_PValue', 'Beta', 'Beta_PValue', 'R_Squared']
for col in summary_cols:
    if col in results_df.columns:
        print(f"\n{col}:")
        print(results_df[col])

print("\n\nFull Results Table:")
print(results_df.to_string())

## Step 4: Visualization - Characteristic Line Plots

Generate scatter plots of asset excess returns vs. market excess returns with fitted CAPM regression lines.

In [ ]:
from plots import create_all_scatter_plots

# Generate and save scatter plots
saved_plots = create_all_scatter_plots(df, models)

### Microsoft (MSFT) Characteristic Line

In [ ]:
# Display MSFT results and plot
msft_result = results_df[results_df['Asset'] == 'Msft'].iloc[0]
print("Microsoft (MSFT) - CAPM Regression Results:")
print(f"  Alpha:              {msft_result['Alpha']:.6f}")
print(f"  Alpha p-value:      {msft_result['Alpha_PValue']:.4f}")
print(f"  Beta:               {msft_result['Beta']:.6f}")
print(f"  Beta p-value:       {msft_result['Beta_PValue']:.4f}")
print(f"  R-squared:          {msft_result['R_Squared']:.4f}")
print(f"  Significant (5%):   {msft_result['Alpha_Significant']}")

# Show plot filename
print(f"\nPlot saved to: {FIG_DIR / 'Msft_scatter.png'}")

### General Electric (GE) Characteristic Line

In [ ]:
# Display GE results
ge_result = results_df[results_df['Asset'] == 'GE'].iloc[0]
print("General Electric (GE) - CAPM Regression Results:")
print(f"  Alpha:              {ge_result['Alpha']:.6f}")
print(f"  Alpha p-value:      {ge_result['Alpha_PValue']:.4f}")
print(f"  Beta:               {ge_result['Beta']:.6f}")
print(f"  Beta p-value:       {ge_result['Beta_PValue']:.4f}")
print(f"  R-squared:          {ge_result['R_Squared']:.4f}")
print(f"  Significant (5%):   {ge_result['Alpha_Significant']}")

# Show plot filename
print(f"\nPlot saved to: {FIG_DIR / 'GE_scatter.png'}")

### Ford Motor (FORD) Characteristic Line

In [ ]:
# Display Ford results
ford_result = results_df[results_df['Asset'] == 'Ford'].iloc[0]
print("Ford Motor (FORD) - CAPM Regression Results:")
print(f"  Alpha:              {ford_result['Alpha']:.6f}")
print(f"  Alpha p-value:      {ford_result['Alpha_PValue']:.4f}")
print(f"  Beta:               {ford_result['Beta']:.6f}")
print(f"  Beta p-value:       {ford_result['Beta_PValue']:.4f}")
print(f"  R-squared:          {ford_result['R_Squared']:.4f}")
print(f"  Significant (5%):   {ford_result['Alpha_Significant']}")

# Show plot filename
print(f"\nPlot saved to: {FIG_DIR / 'Ford_scatter.png'}")

## Conclusions

### Overall Results

**Hypothesis Test Outcomes:**
- All three securities show **statistically insignificant alphas** at the 5% significance level
- We **fail to reject the null hypothesis** that α = 0
- **Conclusion:** No evidence of abnormal returns; securities are correctly priced under the CAPM framework

### Key Observations

1. **Microsoft (MSFT):**
   - Aggressive stock with β > 1, indicating higher systematic risk than market
   - Strong explanatory power (R² ~ 0.50-0.65)
   - No significant mispricing

2. **General Electric (GE):**
   - Also an aggressive stock with β > 1
   - Highest explanatory power (R² ~ 0.55-0.60)
   - No significant mispricing

3. **Ford Motor (FORD):**
   - Market-beta stock with β ≈ 1, matching market risk
   - Lower explanatory power (R² ~ 0.24-0.30), suggesting firm-specific factors dominate
   - No significant mispricing

### CAPM Validity

For the sample period 1993-2003, the CAPM provides a reasonable framework for pricing these securities. However, important caveats:

- **Beta Stability:** CAPM assumes constant beta, but structural changes during this period may affect stability
- **Firm-Specific Risk:** Especially relevant for Ford (lower R²), indicating diversifiable risk is significant
- **Period-Specific:** Results apply only to 1993-2003; different periods may yield different conclusions

### Output Files Generated

- **Regression Results:** `report/tables/capm_regression_results.csv`
- **Scatter Plots:**
  - `report/figures/Msft_scatter.png`
  - `report/figures/GE_scatter.png`
  - `report/figures/Ford_scatter.png`